# Merchant Fraud Risk Profile

This notebook covers **Member 4 fraud risk only**. It does not calculate a business score, tune ranking weights, or produce a Top 100. The primary output is a transparent relative risk index built from consumer exposure and directly observed merchant fraud evidence.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

def find_member4_dir(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        direct = candidate if candidate.name == 'member4_fraud' else candidate / 'member4_fraud'
        if (direct / 'code').is_dir():
            return direct
    raise FileNotFoundError('Run this notebook from the repository or member4_fraud/code.')

MEMBER4_DIR = find_member4_dir()
CODE_DIR = MEMBER4_DIR / 'code'
sys.path.insert(0, str(CODE_DIR))

from build_fraud_risk_profile import build_fraud_risk_profile

OUTPUT_DIR = MEMBER4_DIR / 'result'
summary = build_fraud_risk_profile(output_dir=OUTPUT_DIR)
display(summary)

,metric,value
0,total_merchants,4422.000000
1,merchants_with_consumer_risk,4414.000000
2,merchants_with_direct_fraud,61.000000
3,consumer_and_direct,61.000000
4,consumer_only,4353.000000
5,direct_only,0.000000
6,no_information,8.000000
7,median_consumer_amount_coverage,1.000000
8,mean_consumer_amount_coverage,1.000000
9,median_observed_consumer_label_amount_coverage,0.840039


## 1. Consumer risk exposure

For merchant $m$, consumer exposure is the amount-weighted average consumer risk:

$$C_m = \frac{\sum_{t \in T_m^*} Amount_t \cdot ConsumerRisk_{u(t)}}{\sum_{t \in T_m^*} Amount_t}$$

It is converted to a merchant percentile $C_m^*$. Transaction and amount coverage are retained as reliability indicators but are not included in the score formula.

In [2]:
profile = pd.read_csv(OUTPUT_DIR / 'merchant_fraud_risk_profile.csv')
status_counts = profile['fraud_evidence_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='merchants')
coverage_columns = [
    'consumer_risk_transaction_coverage', 'consumer_risk_amount_coverage',
    'observed_consumer_label_transaction_coverage',
    'observed_consumer_label_amount_coverage'
]
coverage_summary = profile[coverage_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
display(status_counts, coverage_summary)

,status,merchants
0,consumer_only,4353
1,consumer_and_direct,61
2,no_information,8


,count,mean,std,min,10%,25%,50%,75%,90%,max
consumer_risk_transaction_coverage,4414.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
consumer_risk_amount_coverage,4414.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
observed_consumer_label_transaction_coverage,4414.0,0.848155,0.065772,0.000000,0.800000,0.825204,0.837753,0.863397,0.928571,1.0
observed_consumer_label_amount_coverage,4411.0,0.853742,0.067893,0.297025,0.797702,0.824918,0.840039,0.878306,0.954145,1.0


## 2. Direct merchant fraud evidence

For the 61 merchants with direct observations, the primary direct evidence is:

$$D_m = Percentile(MeanObservedMerchantFraud_m)$$

Observation count and maximum observed fraud probability are diagnostics only. They are deliberately excluded from the main formula because the maximum can be dominated by one observation.

In [3]:
direct_columns = [
    'merchant_abn', 'merchant_name', 'merchant_fraud_observation_count',
    'mean_observed_merchant_fraud', 'max_observed_merchant_fraud',
    'direct_merchant_fraud_percentile'
]
direct_profile = profile.loc[profile.has_direct_merchant_fraud_information, direct_columns]
display(direct_profile.sort_values('direct_merchant_fraud_percentile', ascending=False).head(10))

,merchant_abn,merchant_name,merchant_fraud_observation_count,mean_observed_merchant_fraud,max_observed_merchant_fraud,direct_merchant_fraud_percentile
3615,82999039227,None,1.0,0.941347,0.941347,1.000000
1301,35575706403,Tempus Mauris Ltd,1.0,0.910961,0.910961,0.983333
4319,97884414539,Ut Corporation,1.0,0.897992,0.897992,0.966667
213,14530561097,Duis At Inc.,1.0,0.808005,0.808005,0.950000
681,23686790459,None,1.0,0.794543,0.794543,0.933333
427,18737319630,Ut Industries,1.0,0.727307,0.727307,0.916667
3719,85482742429,Mi Eleifend Egestas LLP,1.0,0.708813,0.708813,0.900000
3371,78080443264,Nullam Enim Sed Incorporated,1.0,0.690953,0.690953,0.883333
3326,76968105359,Nec Limited,1.0,0.682784,0.682784,0.866667
3465,80089686333,Tincidunt Nibh LLP,1.0,0.675058,0.675058,0.850000


## 3. Combined fraud risk

The baseline uses equal weights when both sources exist and the sole available source otherwise:

$$FraudRisk_m = \begin{cases}0.5C_m^* + 0.5D_m, & C,D\text{ both available}\\C_m^*, & C\text{ only}\\D_m, & D\text{ only}\\NA, & \text{neither available}\end{cases}$$

Two alternative evidence-weight scenarios (0.7/0.3 and 0.3/0.7) are reported for fraud-method sensitivity only. No merchant ranking is calculated here.

In [4]:
sensitivity = pd.read_csv(OUTPUT_DIR / 'fraud_risk_weight_sensitivity.csv')
sensitivity['difference_consumer70_vs_equal'] = (
    sensitivity['fraud_risk_consumer_70_direct_30'] - sensitivity['fraud_risk_equal_weight']
).abs()
sensitivity['difference_direct70_vs_equal'] = (
    sensitivity['fraud_risk_consumer_30_direct_70'] - sensitivity['fraud_risk_equal_weight']
).abs()
display(sensitivity.describe().T, sensitivity.head())

,count,mean,std,min,25%,50%,75%,max
merchant_abn,61.0,5.892628e+10,3.091745e+10,1.114906e+10,2.709379e+10,7.305252e+10,8.548274e+10,9.998904e+10
consumer_exposure_percentile,61.0,3.755521e-01,2.622201e-01,3.625651e-03,1.257648e-01,3.299343e-01,5.574439e-01,9.956945e-01
direct_merchant_fraud_percentile,61.0,5.000000e-01,2.958783e-01,8.333333e-03,2.500000e-01,5.000000e-01,7.500000e-01,1.000000e+00
fraud_risk_consumer_70_direct_30,61.0,4.128865e-01,1.998603e-01,5.037956e-03,2.575595e-01,3.863483e-01,4.778779e-01,8.821788e-01
fraud_risk_equal_weight,61.0,4.377761e-01,1.927149e-01,5.979492e-03,3.171652e-01,4.030308e-01,5.334240e-01,8.758611e-01
fraud_risk_consumer_30_direct_70,61.0,4.626656e-01,2.178485e-01,6.921029e-03,3.284852e-01,4.192930e-01,6.519884e-01,9.055167e-01
difference_consumer70_vs_equal,61.0,6.867322e-02,4.896149e-02,2.673918e-04,2.987537e-02,6.818038e-02,8.767278e-02,1.974722e-01
difference_direct70_vs_equal,61.0,6.867322e-02,4.896149e-02,2.673918e-04,2.987537e-02,6.818038e-02,8.767278e-02,1.974722e-01


,merchant_abn,merchant_name,consumer_exposure_percentile,direct_merchant_fraud_percentile,fraud_risk_consumer_70_direct_30,fraud_risk_equal_weight,fraud_risk_consumer_30_direct_70,difference_consumer70_vs_equal,difference_direct70_vs_equal
0,11149063370,Et Arcu Limited,0.059823,0.650000,0.236876,0.354912,0.472947,0.118035,0.118035
1,11470993597,Sed Associates,0.423295,0.750000,0.521306,0.586647,0.651988,0.065341,0.065341
2,11590404675,Arcu Sed PC,0.086336,0.250000,0.135435,0.168168,0.200901,0.032733,0.032733
3,14530561097,Duis At Inc.,0.801722,0.950000,0.846206,0.875861,0.905517,0.029656,0.029656
4,14827550074,NaN,0.125085,0.566667,0.257559,0.345876,0.434192,0.088316,0.088316


## 4. Interpretation and limitations

- `fraud_risk_index` is a relative risk index, **not** a fraud probability.
- Only 61 merchants have direct merchant fraud observations.
- Absence from the merchant fraud file does not imply that a merchant is safe.
- Consumer exposure uses model-estimated consumer risk and is not proof of merchant wrongdoing.
- Model-score coverage and raw consumer-label coverage are reported separately; the former can be high even when direct label evidence is incomplete.
- Coverage, direct observation count, and maximum observed risk must accompany the index as reliability diagnostics.
- The eight merchants with neither evidence source remain `NA`; missing evidence is never replaced with zero.